# 03 — Agent 1: Risk Profiler (CatBoost)

**Model**: CatBoost with native categorical support
**Target**: risk_tier (LOW/MEDIUM/HIGH)
**Explainability**: SHAP + LIME + Anchors + DiCE

**Why CatBoost**: Native categorical handling, published results on insurance data, ordered target encoding.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

MODELS_DIR = Path("../models")
train_df = pd.read_parquet(MODELS_DIR / "train.parquet")
test_df = pd.read_parquet(MODELS_DIR / "test.parquet")
feature_config = joblib.load(MODELS_DIR / "feature_config.joblib")

FEATURES = feature_config["AGENT1_FEATURES"]
CAT_FEATURES = feature_config["AGENT1_CAT_FEATURES"]
RISK_TIER_MAP = feature_config["RISK_TIER_MAP"]
TIER_NAMES = ["LOW", "MEDIUM", "HIGH"]
TARGET = "risk_tier"

train_df["risk_tier_label"] = train_df[TARGET].map(RISK_TIER_MAP)
test_df["risk_tier_label"] = test_df[TARGET].map(RISK_TIER_MAP)

for col in CAT_FEATURES:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

X_train = train_df[FEATURES]
y_train = train_df["risk_tier_label"]
X_test = test_df[FEATURES]
y_test = test_df["risk_tier_label"]

cat_indices = [FEATURES.index(c) for c in CAT_FEATURES]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Cat features: {CAT_FEATURES} at indices {cat_indices}")
print(f"Class distribution (train):\n{y_train.value_counts().sort_index()}")

## 1. Train CatBoost

In [ ]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, confusion_matrix

train_pool = Pool(X_train, y_train, cat_features=cat_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_indices)

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    task_type="CPU",
)

model.fit(train_pool, eval_set=test_pool)

y_pred = model.predict(X_test).flatten().astype(int)

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=TIER_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=TIER_NAMES, yticklabels=TIER_NAMES, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Agent 1 — CatBoost Confusion Matrix")
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent1_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAccuracy: {(y_pred == y_test).mean():.4f}")

## 2. Explainability: SHAP (Feature Importance)

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_by_class = shap_values
else:
    shap_by_class = [shap_values[:, :, i] for i in range(shap_values.shape[2])]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, tier in enumerate(TIER_NAMES):
    plt.sca(axes[i])
    shap.summary_plot(shap_by_class[i], X_sample, show=False, plot_size=None)
    axes[i].set_title(f"SHAP — {tier}")
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent1_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

sample_row = X_test.iloc[[0]]
pred_class = int(model.predict(sample_row).flatten()[0])
sv_single = explainer.shap_values(sample_row)
if isinstance(sv_single, list):
    class_shap = sv_single[pred_class][0]
else:
    class_shap = sv_single[0, :, pred_class]
paired = sorted(zip(FEATURES, class_shap), key=lambda p: abs(p[1]), reverse=True)
print(f"\nSingle prediction \u2192 {TIER_NAMES[pred_class]}")
print("Top SHAP features:")
for name, val in paired[:5]:
    print(f"  {name}: {val:+.4f}")

## 3. Explainability: LIME (Local Linear Approximation)

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

X_train_numeric = X_train.copy()
X_test_numeric = X_test.copy()
cat_value_maps = {}
for col in CAT_FEATURES:
    cats = sorted(X_train[col].unique())
    cat_map = {v: i for i, v in enumerate(cats)}
    cat_value_maps[col] = {i: v for v, i in cat_map.items()}
    X_train_numeric[col] = X_train[col].map(cat_map).fillna(0).astype(float)
    X_test_numeric[col] = X_test[col].map(cat_map).fillna(0).astype(float)

lime_explainer = LimeTabularExplainer(
    X_train_numeric.values.astype(float),
    feature_names=FEATURES,
    class_names=TIER_NAMES,
    categorical_features=cat_indices,
    mode="classification",
    random_state=42,
)

def predict_for_numeric(X_numeric):
    df_temp = pd.DataFrame(X_numeric, columns=FEATURES)
    for col in CAT_FEATURES:
        reverse_map = cat_value_maps[col]
        df_temp[col] = df_temp[col].round().astype(int).map(reverse_map).fillna(list(reverse_map.values())[0])
    pool = Pool(df_temp, cat_features=cat_indices)
    return model.predict_proba(pool)

sample_idx = 0
lime_exp = lime_explainer.explain_instance(
    X_test_numeric.iloc[sample_idx].values.astype(float),
    predict_for_numeric,
    num_features=6,
    top_labels=1,
)

pred_label = int(model.predict(X_test.iloc[[sample_idx]]).flatten()[0])
available_labels = list(lime_exp.local_exp.keys())
label_to_show = pred_label if pred_label in available_labels else available_labels[0]
print(f"Predicted: {TIER_NAMES[pred_label]}")
print(f"\nLIME explanation (top features):")
for feat, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feat}: {weight:+.4f}")

fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.title(f"LIME — Predicted: {TIER_NAMES[pred_label]}")
plt.tight_layout()
plt.savefig(MODELS_DIR / "agent1_lime_example.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Explainability: Anchors (IF-THEN Rules)

Anchor explanations produce human-readable rules with precision and coverage metrics — the format underwriters naturally think in.

In [ ]:
from alibi.explainers import AnchorTabular

def anchor_predict_fn(X):
    results = []
    for row in X:
        df_temp = pd.DataFrame([row], columns=FEATURES)
        for col in CAT_FEATURES:
            reverse_map = cat_value_maps[col]
            df_temp[col] = df_temp[col].round().astype(int).map(reverse_map).fillna(list(reverse_map.values())[0])
        pool = Pool(df_temp, cat_features=cat_indices)
        pred = int(model.predict(pool).flatten()[0])
        results.append(pred)
    return np.array(results)

anchor_explainer = AnchorTabular(
    predictor=anchor_predict_fn,
    feature_names=FEATURES,
    categorical_names={i: sorted(X_train[col].unique()) for i, col in enumerate(FEATURES) if col in CAT_FEATURES},
)

anchor_explainer.fit(X_train_numeric.values.astype(float), disc_perc=(25, 50, 75))

sample_idx = 0
explanation = anchor_explainer.explain(X_test_numeric.iloc[sample_idx].values.astype(float))

pred_label = int(model.predict(X_test.iloc[[sample_idx]]).flatten()[0])
anchor_rule = " AND ".join(explanation.anchor) if explanation.anchor else "No stable rule found"
print(f"Predicted: {TIER_NAMES[pred_label]}")
print(f"\nAnchor Rule: {anchor_rule}")
print(f"Precision: {explanation.precision:.4f}")
print(f"Coverage:  {explanation.coverage:.4f}")

## 5. Explainability: DiCE Counterfactuals

DiCE generates "what-if" scenarios: what minimal changes to input features would change the risk tier? This is the most actionable explanation for underwriters.

In [ ]:
import dice_ml


class CatBoostWrapper:
    def __init__(self, model, cat_indices, features, cat_value_maps):
        self.model = model
        self.cat_indices = cat_indices
        self.features = features
        self.cat_value_maps = cat_value_maps

    def _to_catboost_df(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=self.features)
        df_temp = X.copy()
        for col in [self.features[i] for i in self.cat_indices]:
            reverse_map = self.cat_value_maps[col]
            df_temp[col] = df_temp[col].round().astype(int).map(reverse_map).fillna(list(reverse_map.values())[0])
        return df_temp

    def predict_proba(self, X):
        df_temp = self._to_catboost_df(X)
        pool = Pool(df_temp, cat_features=self.cat_indices)
        return self.model.predict_proba(pool)

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)


wrapper = CatBoostWrapper(model, cat_indices, FEATURES, cat_value_maps)

train_dice = X_train_numeric.copy()
train_dice["risk_tier_label"] = y_train.values

d = dice_ml.Data(
    dataframe=train_dice,
    continuous_features=[f for f in FEATURES if f not in CAT_FEATURES],
    outcome_name="risk_tier_label",
)

m = dice_ml.Model(model=wrapper, backend="sklearn", model_type="classifier")
dice_exp = dice_ml.Dice(d, m, method="random")

sample_row = X_test_numeric.iloc[[0]]
pred_class = int(model.predict(X_test.iloc[[0]]).flatten()[0])
desired_class = 0 if pred_class != 0 else 2

print(f"Original prediction: {TIER_NAMES[pred_class]}")
print(f"Desired class: {TIER_NAMES[desired_class]}")
try:
    cf = dice_exp.generate_counterfactuals(
        sample_row,
        total_CFs=3,
        desired_class=desired_class,
    )
    print(f"\nCounterfactual examples (what would change the risk tier):")
    cf.visualize_as_dataframe(show_only_changes=True)
except Exception as e:
    print(f"DiCE could not generate counterfactuals for this sample: {e}")

## 6. Save Model & Explainers

In [ ]:
joblib.dump(model, MODELS_DIR / "risk_model.joblib")
joblib.dump(explainer, MODELS_DIR / "risk_explainer.joblib")
joblib.dump(cat_value_maps, MODELS_DIR / "risk_cat_value_maps.joblib")
print("Saved: risk_model.joblib, risk_explainer.joblib, risk_cat_value_maps.joblib")
print(f"Accuracy: {(y_pred == y_test).mean():.4f}")